# Train XGBoost liquidation risk model

Requires `data/synthetic_aave_data.csv` (run `python -m src.ml.generate_dataset` from repo root first).

Run this notebook with the working directory set to the repository root (recommended).

In [1]:
from pathlib import Path

import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

cwd = Path.cwd()
repo_root = cwd if (cwd / "data").exists() else cwd.parent
csv_path = repo_root / "data" / "synthetic_aave_data.csv"
model_dir = repo_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)
FEATURE_COLS = ["current_health_factor", "debt_to_collateral_ratio", "recent_borrow_count"]
X = df[FEATURE_COLS]
y = df["is_liquidated"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    eval_metric="logloss",
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

out_path = model_dir / "xgboost_risk_model.json"
clf.save_model(str(out_path))
print("Saved model to", out_path.resolve())

              precision    recall  f1-score   support

           0     0.7620    0.8471    0.8023      1164
           1     0.7479    0.6316    0.6848       836

    accuracy                         0.7570      2000
   macro avg     0.7549    0.7393    0.7436      2000
weighted avg     0.7561    0.7570    0.7532      2000

ROC-AUC: 0.8146554736184417
Saved model to C:\Dev\defi-streaming-risk\models\xgboost_risk_model.json
